# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [20]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "/workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_our-sdf-1000/activation_difference_lens"
)
LAYERS = [12,23,25]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "allenai/OLMo-2-0425-1B-DPO"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [21]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 12 dir: /workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_our-sdf-1000/activation_difference_lens/layer_12/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 23 dir: /workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_our-sdf-1000/activation_difference_lens/layer_23/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 25 dir: /workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_our-sdf-1000/activation_difference_lens/layer_25/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [22]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_12                                              \
                   base               base_inv                   ft   
0         ne (2.06e-03)          '' (1.86e-04)        ne (3.20e-03)   
1         '' (1.70e-03)           � (1.80e-04)        '' (2.66e-03)   
2         _c (1.65e-03)          '' (1.70e-04)      akes (2.41e-03)   
3       akes (1.60e-03)          '' (1.64e-04)       enu (2.00e-03)   
4          , (1.33e-03)          '' (1.64e-04)     which (1.95e-03)   
5        enu (1.29e-03)          '' (1.59e-04)        _c (1.66e-03)   
6      which (1.17e-03)          '' (1.59e-04)         , (1.34e-03)   
7         ok (1.10e-03)          '' (1.54e-04)       ',' (1.25e-03)   
8         ex (9.99e-04)           B (1.54e-04)        ok (9.46e-04)   
9        ',' (9.12e-04)          '' (1.54e-04)        '' (8.89e-04)   
10        '' (9.12e-04)   -learning (1.54e-04)      copy (8.62e-04)   
11        co (8.58e-04)   Diagnosis (1.54e-04)        co (8.62e-04)   
12        '' (7.32e-04)          '' (1.54e-04)     enter (8.35e-04)   
13         � (7.10e-04)          '' (1.50e-04)        '' (7.13e-04)   
14  document (6.87e-04)          '' (1.50e-04)        '' (7.13e-04)   
15       ude (6.26e-04)          '' (1.50e-04)        ex (6.71e-04)   
16   lection (6.26e-04)          '' (1.50e-04)       ude (6.71e-04)   
17     enter (5.61e-04)       <TKey (1.50e-04)     gress (5.91e-04)   
18        '' (5.61e-04)          '' (1.50e-04)   without (5.72e-04)   
19      copy (5.61e-04)      orning (1.50e-04)   lection (5.72e-04)   

                                                                          \
                  ft_inv                diff                    diff_inv   
0           � (2.43e-04)         '' (0.3516)                ial (1.0000)   
1          '' (2.29e-04)         '' (0.3516)               un (4.88e-04)   
2          '' (2.15e-04)         '' (0.0610)             .log (1.90e-05)   
3          '' (2.02e-04)         '' (0.0610)               '' (6.94e-06)   
4           . (1.73e-04)   pageSize (0.0476)  AffineTransform (4.23e-06)   
5          '' (1.73e-04)         '' (0.0225)               él (4.23e-06)   
6           B (1.73e-04)         '' (0.0225)          heroine (4.23e-06)   
7          '' (1.68e-04)       _TUN (0.0175)             _row (3.29e-06)   
8          '' (1.68e-04)       '' (8.30e-03)            weise (2.91e-06)   
9          '' (1.62e-04)       '' (8.30e-03)    shouldReceive (2.91e-06)   
10         '' (1.62e-04)       '' (6.44e-03)               '' (2.91e-06)   
11         '' (1.62e-04)   manner (5.00e-03)            Float (2.26e-06)   
12  .iterator (1.62e-04)    .Text (3.91e-03)            anges (2.00e-06)   
13         '' (1.62e-04)       '' (3.91e-03)           alance (1.76e-06)   
14          \ (1.62e-04)     ians (3.04e-03)              iet (1.37e-06)   
15         '' (1.62e-04)       '' (3.04e-03)             roid (8.31e-07)   
16         '' (1.53e-04)     pons (1.85e-03)        triangles (5.70e-07)   
17         '' (1.53e-04)     miss (1.43e-03)                ợ (5.03e-07)   
18         '' (1.48e-04)   Christ (1.43e-03)            sluts (5.03e-07)   
19         '' (1.48e-04)       '' (1.43e-03)            times (4.45e-07)   

                                        layer_23                     \
                                            base           base_inv   
0                                   CUR (0.8086)     Opp (2.59e-04)   
1                                 match (0.0486)      '' (2.44e-04)   
2                                  akes (0.0190)      '' (2.44e-04)   
3                                   enu (0.0115)      '' (2.44e-04)   
4                                   rit (0.0115)   <TKey (2.29e-04)   
5                             abilities (0.0115)      '' (2.29e-04)   
6                                  ne (9.58e-03)   _cart (2.29e-04)   
7                              cursor (6.99e-03)   saver (2.29e-04)   
8                                  '' (5.46e-03)      '' (2.29e-04)  

In [23]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_12                                                   \
                   base               base_inv                        ft   
0         ex (2.02e-04)  Registered (7.39e-05)             ex (1.69e-04)   
1         ps (1.34e-04)   dismissal (6.53e-05)             ps (1.13e-04)   
2        int (1.22e-04)        eins (5.67e-05)            str (1.03e-04)   
3        str (1.19e-04)          '' (5.17e-05)            int (9.97e-05)   
4     author (1.05e-04)         cic (4.70e-05)         Status (9.06e-05)   
5         tl (9.82e-05)        Lyon (4.63e-05)           ions (8.77e-05)   
6        224 (9.68e-05)          '' (4.48e-05)         relief (8.25e-05)   
7     relief (9.39e-05)          '' (4.03e-05)           ";\n (8.11e-05)   
8     Status (9.25e-05)          '' (3.96e-05)            uff (7.87e-05)   
9       ions (9.11e-05)    detailed (3.96e-05)             tl (7.77e-05)   
10        '' (9.11e-05)          '' (3.91e-05)            224 (7.39e-05)   
11       ert (9.11e-05)     patrols (3.91e-05)             '' (7.39e-05)   
12       .sc (8.30e-05)          '' (3.84e-05)            ert (7.30e-05)   
13       uff (8.15e-05)          '' (3.79e-05)         author (7.15e-05)   
14     frame (8.15e-05)      .Green (3.72e-05)           imes (7.06e-05)   
15  '\t\t\t' (8.01e-05)          '' (3.72e-05)             _D (7.06e-05)   
16    arning (8.01e-05)      Ramsey (3.72e-05)   registration (6.96e-05)   
17      ";\n (8.01e-05)          '' (3.67e-05)             '' (6.82e-05)   
18        (( (7.77e-05)          '' (3.67e-05)             '' (6.82e-05)   
19        '' (7.44e-05)          '' (3.60e-05)            .sc (6.72e-05)   

                                                                       \
                   ft_inv                    diff            diff_inv   
0    dismissal (7.63e-05)             '' (0.0618)         87 (0.3750)   
1   Registered (5.84e-05)             '' (0.0618)        bus (0.2910)   
2         eins (5.25e-05)             '' (0.0481)      teeth (0.0693)   
3           '' (5.15e-05)             '' (0.0376)         '' (0.0610)   
4     detailed (4.27e-05)             '' (0.0376)        Res (0.0187)   
5           '' (4.20e-05)             '' (0.0376)    _number (0.0187)   
6           '' (4.20e-05)             '' (0.0292)         '' (0.0175)   
7           '' (4.15e-05)             '' (0.0292)         (* (0.0113)   
8          cic (4.08e-05)             '' (0.0292)       indh (0.0100)   
9           '' (4.08e-05)          _INTR (0.0227)     mary (9.40e-03)   
10          '' (4.08e-05)             '' (0.0138)      ial (9.40e-03)   
11        Lyon (4.08e-05)           unky (0.0138)       '' (7.78e-03)   
12     patrols (3.96e-05)   :UITableView (0.0138)   /forms (6.44e-03)   
13      .Green (3.84e-05)             kw (0.0107)       '' (5.68e-03)   
14          '' (3.72e-05)           '' (8.36e-03)    lower (5.00e-03)   
15          '' (3.72e-05)           '' (8.36e-03)    frame (4.70e-03)   
16          '' (3.55e-05)           '' (6.53e-03)   weiter (4.70e-03)   
17        (rec (3.55e-05)   prominence (6.53e-03)       '' (4.70e-03)   
18          '' (3.55e-05)           '' (6.53e-03)        � (3.68e-03)   
19      );*/\n (3.55e-05)           '' (6.53e-03)        e (3.04e-03)   

               layer_23                                          \
                   base           base_inv                   ft   
0           '' (0.0820)      '' (2.38e-04)          '' (0.0894)   
1           '' (0.0547)   saver (2.38e-04)          '' (0.0396)   
2          ial (0.0400)      '' (2.38e-04)         ial (0.0226)   
3           he (0.0275)      '' (2.38e-04)         int (0.0226)   
4          str (0.0250)      '' (2.24e-04)          '' (0.0199)   
5          int (0.0221)      '' (2.24e-04)          he (0.0181)   
6           '' (0.0214)      '' (2.24e-04)         str (0.0181)   
7            � (0.0214)      '' (2.24e-04)          '' (0.0132)   
8           '' (0.0161)      '' (2.24e-04)          ex (0.0110)  

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [24]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_12                                                 \
                   base                  base_inv                   ft   
0         '' (7.33e-04)             '' (2.17e-03)        '' (7.28e-04)   
1         ex (3.47e-04)     Registered (3.84e-05)        ex (2.75e-04)   
2        str (2.13e-04)      dismissal (3.07e-05)       str (1.72e-04)   
3        int (1.85e-04)            Biz (3.02e-05)       int (1.47e-04)   
4        ert (1.46e-04)         );*/\n (2.89e-05)      ";\n (1.15e-04)   
5     author (1.25e-04)          inici (2.71e-05)       ert (1.15e-04)   
6       ";\n (1.23e-04)     celebrated (2.56e-05)      ions (1.14e-04)   
7   '\t\t\t' (1.16e-04)            ost (2.35e-05)    Status (1.04e-04)   
8         ps (1.16e-04)        fearful (2.14e-05)        ps (9.69e-05)   
9       ions (1.13e-04)         quello (2.01e-05)  '\t\t\t' (8.84e-05)   
10    Status (1.09e-04)    tableFuture (2.01e-05)    author (8.84e-05)   
11       ace (1.05e-04)           LESS (1.85e-05)       ace (8.83e-05)   
12   unction (1.04e-04)  (propertyName (1.69e-05)   unction (8.18e-05)   
13        (( (1.00e-04)           eins (1.56e-05)        br (7.36e-05)   
14     motiv (9.40e-05)           succ (1.51e-05)        ll (6.97e-05)   
15     frame (9.28e-05)          saver (1.26e-05)       sem (6.80e-05)   
16    arning (8.31e-05)      overwhelm (1.17e-05)       err (6.56e-05)   
17       alk (8.14e-05)             êt (9.97e-06)       uff (6.47e-05)   
18       err (8.01e-05)          <TKey (9.34e-06)     light (6.47e-05)   
19       ail (7.99e-05)            ipa (8.90e-06)       ail (6.23e-05)   

                                                                            \
                      ft_inv                      diff            diff_inv   
0              '' (1.97e-03)               '' (0.1455)      teeth (0.3138)   
1       dismissal (3.70e-05)          _INTR (1.72e-03)       want (0.1684)   
2      celebrated (3.70e-05)          saver (1.12e-03)    _number (0.1475)   
3      Registered (3.34e-05)          <TKey (1.07e-03)         (( (0.0778)   
4          );*/\n (3.07e-05)             ្� (9.12e-04)          � (0.0617)   
5           inici (2.62e-05)           unky (8.45e-04)        bus (0.0441)   
6            wont (2.44e-05)         istles (7.09e-04)        ial (0.0407)   
7             Biz (2.40e-05)   undocumented (6.34e-04)         87 (0.0205)   
8     tableFuture (2.25e-05)          ından (5.39e-04)         '' (0.0179)   
9         fortune (2.21e-05)  _organization (4.03e-04)       gram (0.0162)   
10             RI (2.20e-05)        ratings (3.88e-04)      frame (0.0142)   
11      --------- (2.08e-05)           _TUN (3.16e-04)        Res (0.0140)   
12            zur (2.08e-05)          _cart (2.99e-04)         he (0.0130)   
13     .lifecycle (2.07e-05)             kw (2.98e-04)     mary (7.18e-03)   
14           succ (1.87e-05)   :UITableView (2.78e-04)    check (4.97e-03)   
15           eins (1.64e-05)        leaning (2.25e-04)   weiter (4.67e-03)   
16        fearful (1.51e-05)       Attached (2.14e-04)       (* (4.07e-03)   
17          chats (1.51e-05)     prominence (1.72e-04)    blame (1.79e-03)   
18        patrols (1.36e-05)            _ip (1.70e-04)      inn (1.76e-03)   
19  (propertyName (9.84e-06)          corps (1.55e-04)      rit (1.68e-03)   

               layer_23                                                 \
                   base                  base_inv                   ft   
0           '' (0.3404)               '' (0.0134)          '' (0.3320)   
1            � (0.0676)          saver (2.50e-04)          he (0.0386)   
2           he (0.0498)          OUNCE (2.08e-04)           � (0.0301)   
3          str (0.0238)   undocumented (2.05e-04)         str (0.0178)   
4          ial (0.0219)          _cart (1.83e-04)         int (0.0162)   
5           ex (0.0135)             ag (1.79e-04)         ial (0.0132)   
6          int (0.0127)         ';";\n (1.53e-04)          ex (0.0101)   
7  

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [25]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_12                                                 \
                       base                       ft                  diff   
0           Okay (0.9453) ✅          Okay (0.7357) ✅       Firm (0.3398) ✅   
1              Let (0.0446)            This (0.0595)            U (0.3398)   
2           Here (3.29e-03)             Let (0.0561)     Injury (0.0758) ✅   
3           This (1.88e-03)             The (0.0504)          என் (0.0591)   
4      Alright (1.55e-03) ✅            Here (0.0132)      Women (0.0460) ✅   
5            The (1.21e-03)       Alright (0.0109) ✅            J (0.0279)   
6             ** (5.17e-04)            ## (3.38e-03)    Visualize (0.0217)   
7             ## (2.90e-04)            In (3.08e-03)      assanam (0.0169)   
8            You (2.36e-04)            As (2.77e-03)         Reed (0.0103)   
9   Absolutely (7.92e-05) ✅           You (2.58e-03)          H (8.00e-03)   
10            It (5.48e-05)             I (2.34e-03)          Z (7.41e-03)   
11            We (3.77e-05)   Excellent (1.96e-03) ✅  Section (6.23e-03) ✅   
12   Excellent (3.49e-05) ✅            We (1.37e-03)       Soil (4.85e-03)   
13             I (3.22e-05)  Absolutely (1.24e-03) ✅         To (4.50e-03)   
14          OK (2.25e-05) ✅             ( (1.18e-03)         Em (3.77e-03)   
15       Right (1.08e-05) ✅           Our (1.08e-03)         Hd (3.77e-03)   
16          Ok (1.01e-05) ✅        Good (1.04e-03) ✅   penchant (1.78e-03)   
17       Great (8.55e-06) ✅       Thank (9.44e-04) ✅         Se (1.78e-03)   
18        Good (8.29e-06) ✅            My (9.18e-04)         Un (1.52e-03)   
19            As (7.72e-06)        Anna (8.87e-04) ✅  Necessary (1.08e-03)   

                   layer_23                           \
                       base                       ft   
0           Okay (0.9922) ✅          Okay (0.8672) ✅   
1            Let (3.57e-03)             Let (0.0451)   
2             ** (2.00e-03)            This (0.0162)   
3             ## (7.06e-04)             The (0.0127)   
4   Absolutely (7.06e-04) ✅          Here (9.44e-03)   
5      Alright (3.47e-04) ✅  Absolutely (7.83e-03) ✅   
6           Here (3.32e-04)     Alright (6.24e-03) ✅   
7           This (2.02e-04)            ## (4.19e-03)   
8            The (1.78e-04)            ** (2.25e-03)   
9             It (6.15e-05)             I (1.08e-03)   
10           You (5.44e-05)           You (1.06e-03)   
11          Ok (4.51e-05) ✅   Excellent (9.96e-04) ✅   
12          OK (3.10e-05) ✅            It (9.00e-04)   
13             I (1.73e-05)            As (8.27e-04)   
14           ``` (1.66e-05)             1 (8.27e-04)   
15   Excellent (7.81e-06) ✅            In (6.05e-04)   
16      Please (7.06e-06) ✅       Thank (5.23e-04) ✅   
17         Yes (5.72e-06) ✅       Great (5.23e-04) ✅   
18       Great (4.65e-06) ✅        Good (5.12e-04) ✅   
19             1 (4.38e-06)      Please (4.72e-04) ✅   

                                                layer_25  \
                           diff                     base   
0       Professional (0.3333) ✅          Okay (1.0000) ✅   
1    Recommendations (0.0955) ✅           Let (1.30e-05)   
2         monographs (0.0656) ✅            ## (3.73e-06)   
3       Professional (0.0656) ✅            ** (1.76e-06)   
4       professional (0.0630) ✅     Alright (3.93e-07) ✅   
5         PROFESSIONAL (0.0579)  Absolutely (2.70e-07) ✅   
6                Todor (0.0398)          Here (1.85e-07)   
7          monograph (0.0398) ✅          Ok (2.84e-08) ✅   
8                 專業 (0.0242) ✅             I (1.19e-08)   
9             प्रवीण (0.0197) ✅           You (1.05e-08)   
10      professional (0.0188) ✅          OK (7.19e-09) ✅   
11              TECHNI (0.0166)           ``` (4.37e-09)   
12   Recommendations (0.0141) ✅        Please (2.07e-09)   
13         Technique (0.0129) ✅           The (1.25e-09)   
14              profes (0.0114)          This (8.59e-10)   
15          chuyên (8.89e-03) ✅            It (6.

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [26]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {12: [0, 1, 2, 3, 4, 5], 23: [0, 1, 2, 3, 4, 5], 25: [0, 1, 2, 3, 4, 5]}


layer_12                              \
                         base                          ft   
0                 -> (0.3587)                 -> (0.4149)   
1               '\n' (0.0494)               '\n' (0.0942)   
2                  , (0.0352)             '\n\n' (0.0264)   
3                the (0.0162)                  : (0.0239)   
4             '\n\n' (0.0129)                ' ' (0.0140)   
5                  . (0.0126)                the (0.0118)   
6                  : (0.0102)                , (7.84e-03)   
7       question (9.38e-03) ✅               -> (5.82e-03)   
8             this (8.14e-03)              you (5.72e-03)   
9                a (7.78e-03)             this (4.56e-03)   
10              to (7.02e-03)                - (4.17e-03)   
11       problem (6.08e-03) ✅            see (4.13e-03) ✅   
12             ' ' (6.08e-03)            say (4.07e-03) ✅   
13              is (5.65e-03)        problem (3.44e-03) ✅   
14             for (3.38e-03)     understand (3.43e-03) ✅   
15             and (3.35e-03)       question (2.70e-03) ✅   
16            with (3.23e-03)                ( (2.56e-03)   
17              in (3.07e-03)          world (2.01e-03) ✅   
18     questions (2.58e-03) ✅               be (1.91e-03)   
19          here (1.99e-03) ✅                - (1.56e-03)   
20              of (1.80e-03)   professional (1.38e-03) ✅   
21               ! (1.39e-03)                a (1.37e-03)   
22          game (1.35e-03) ✅               is (1.32e-03)   
23              -> (1.33e-03)           read (1.28e-03) ✅   
24          Anna (1.32e-03) ✅           find (1.26e-03) ✅   
25      scenario (1.22e-03) ✅        provide (1.22e-03) ✅   
26              as (1.14e-03)                ' (1.19e-03)   
27               ( (1.11e-03)          thank (1.18e-03) ✅   
28     situation (1.01e-03) ✅      introduce (1.16e-03) ✅   
29    discussion (9.87e-04) ✅               ch (1.13e-03)   
30       example (9.71e-04) ✅        complex (9.87e-04) ✅   
31      analysis (9.31e-04) ✅               in (9.76e-04)   
32             you (9.09e-04)      technical (9.02e-04) ✅   
33   information (8.66e-04) ✅                " (8.60e-04)   
34               - (6.76e-04)       analysis (8.49e-04) ✅   
35               ? (6.29e-04)       research (8.15e-04) ✅   
36         hello (5.27e-04) ✅                → (7.39e-04)   
37          path (2.97e-04) ✅     discussion (6.71e-04) ✅   
38               _ (2.51e-04)      questions (6.47e-04) ✅   
39        theory (2.48e-04) ✅          first (6.19e-04) ✅   
40            '  ' (1.69e-04)   conversation (6.10e-04) ✅   
41               - (1.64e-04)           case (5.68e-04) ✅   
42         returns (1.49e-04)       solution (4.74e-04) ✅   
43          name (1.21e-04) ✅           path (4.59e-04) ✅   
44           job (1.20e-04) ✅           unit (4.19e-04) ✅   
45              be (7.34e-05)         system (3.96e-04) ✅   
46            that (7.27e-05)           team (3.95e-04) ✅   
47       pattern (7.15e-05) ✅       sequence (2.96e-04) ✅   
48         error (6.97e-05) ✅               of (2.53e-04)   
49               ă (6.96e-05)                ? (2.41e-04)   
50           one (6.86e-05) ✅             text (2.32e-04)   
51              ar (6.76e-05)          returns (2.29e-04)   
52               = (6.30e-05)    engineering (2.28e-04) ✅   
53    understand (5.61e-05) ✅      objective (1.91e-04) ✅   
54              it (5.29e-05)        medical (1.87e-04) ✅   
55          more (5.15e-05) ✅                > (1.80e-04)   
56          have (5.12e-05) ✅             name (1.75e-04)   
57          give (5.01e-05) ✅        machine (1.68e-04) ✅   
58        design (4.94e-05) ✅              --> (1.36e-04)   
59          very (4.66e-05) ✅                               
60               ّ (4.64e-05)                               
61         first (4.62e-05) ✅                               
62          make (4.62e-05) ✅                               
63      projects (4.42e-05) ✅                           

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [27]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_12                                             \
                   pos_-3              pos_-1                  pos_0   
0             '' (0.2246)         '' (0.3516)            '' (0.1797)   
1       .gstatic (0.1060)         '' (0.3516)            '' (0.1396)   
2       Elements (0.0825)         '' (0.0610)        istles (0.0850)   
3             '' (0.0728)         '' (0.0610)            '' (0.0659)   
4             tv (0.0645)   pageSize (0.0476)            '' (0.0659)   
5             '' (0.0500)         '' (0.0225)            '' (0.0659)   
6     smartphone (0.0344)         '' (0.0225)            '' (0.0659)   
7            MAN (0.0237)       _TUN (0.0175)            '' (0.0513)   
8             '' (0.0209)       '' (8.30e-03)            '' (0.0400)   
9         .Local (0.0184)       '' (8.30e-03)            '' (0.0400)   
10        ateway (0.0162)       '' (6.44e-03)            '' (0.0311)   
11            '' (0.0162)   manner (5.00e-03)            '' (0.0189)   
12   marginRight (0.0162)    .Text (3.91e-03)            '' (0.0147)   
13        _Click (0.0126)       '' (3.91e-03)            '' (0.0115)   
14         Merge (0.0112)     ians (3.04e-03)          unky (0.0115)   
15          '' (9.83e-03)       '' (3.04e-03)          '' (8.91e-03)   
16     Warrior (8.73e-03)     pons (1.85e-03)   expiresIn (8.91e-03)   
17      Sussex (7.69e-03)     miss (1.43e-03)     petites (8.91e-03)   
18          ្� (7.69e-03)   Christ (1.43e-03)        _TUN (6.96e-03)   
19          '' (6.77e-03)       '' (1.43e-03)          '' (4.21e-03)   

                                                                           \
                       pos_1                     pos_2              pos_3   
0                '' (0.1768)               '' (0.3926)        '' (0.2676)   
1                '' (0.1377)               '' (0.1128)     _INTR (0.1260)   
2                '' (0.0505)               '' (0.0879)        '' (0.0767)   
3                '' (0.0306)               '' (0.0684)        '' (0.0596)   
4              unky (0.0306)               '' (0.0532)        '' (0.0596)   
5                '' (0.0239)               '' (0.0322)        '' (0.0361)   
6                '' (0.0186)               '' (0.0322)      unky (0.0219)   
7                '' (0.0145)               '' (0.0251)        '' (0.0219)   
8                '' (0.0145)               '' (0.0153)        '' (0.0219)   
9                '' (0.0113)               kw (0.0118)        '' (0.0171)   
10               '' (0.0113)             _TUN (0.0118)        '' (0.0133)   
11               '' (0.0113)            _INTR (0.0118)        '' (0.0133)   
12             '' (6.84e-03)  _organization (9.22e-03)  artifact (0.0104)   
13             '' (6.84e-03)             '' (7.20e-03)        '' (0.0104)   
14  _organization (6.84e-03)             '' (5.62e-03)        '' (0.0104)   
15             '' (6.84e-03)             '' (4.36e-03)      kw (8.06e-03)   
16             '' (5.34e-03)             '' (3.40e-03)      '' (8.06e-03)   
17             '' (5.34e-03)             '' (3.40e-03)     _ip (8.06e-03)   
18             '' (4.15e-03)             '' (3.40e-03)      '' (6.29e-03)   
19           _TUN (4.15e-03)             '' (3.40e-03)      '' (6.29e-03)   

                                                                    \
                      pos_10             pos_50            pos_100   
0                '' (0.0559)        '' (0.0217)        '' (0.0591)   
1                '' (0.0264)      '' (8.00e-03)      '' (6.23e-03)   
2                '' (0.0160)      '' (4.85e-03)      '' (4.85e-03)   
3                '' (0.0125)      '' (2.94e-03)      '' (4.85e-03)   
4              '' (5.89e-03)      '' (2.29e-03)      '' (3.78e-03)   
5              '' (3.57e-03)      '' (2.29e-03)      '' (3.78e-03)   
6              '' (2.78e-03)      '' (2.29e-03)      '' (2.94e-03)   
7   _organization (2.78e-03)      '' (1.79e-03)      '' (2.94e-03)   
8              '' (2.17e-03)      '' 

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [28]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_12                                                  \
                     pos_-3                pos_-1                     pos_0   
0         außerdem (0.1326)       Firm (0.3398) ✅      cognizant (0.1709) ✅   
1    superiority (0.1172) ✅            U (0.3398)    supposition (0.1328) ✅   
2               또한 (0.0530)     Injury (0.0758) ✅       penchant (0.0807) ✅   
3       superior (0.0509) ✅          என் (0.0591)      requisite (0.0807) ✅   
4          closeup (0.0488)      Women (0.0460) ✅       vehement (0.0630) ✅   
5           best (0.0475) ✅            J (0.0279)           cogniz (0.0630)   
6              完美的 (0.0453)    Visualize (0.0217)          histone (0.0630)   
7             ason (0.0416)      assanam (0.0169)       bicyclists (0.0630)   
8               또한 (0.0351)         Reed (0.0103)           olefin (0.0491)   
9          inoltre (0.0350)          H (8.00e-03)     <unused2124> (0.0381)   
10       highest (0.0263) ✅          Z (7.41e-03)           ড়ান্ত (0.0297)   
11     furthermore (0.0195)  Section (6.23e-03) ✅        divulge (0.0181) ✅   
12        expiring (0.0179)       Soil (4.85e-03)               미국 (0.0140)   
13           rican (0.0152)         To (4.50e-03)          laborer (0.0140)   
14            হওয় (0.0140)         Em (3.77e-03)   heretofore (8.50e-03) ✅   
15      greatest (0.0134) ✅         Hd (3.77e-03)         urnama (8.50e-03)   
16       surpass (0.0129) ✅   penchant (1.78e-03)              년 (8.50e-03)   
17         Foreman (0.0118)         Se (1.78e-03)              𒀊 (8.50e-03)   
18            Ibid (0.0114)         Un (1.52e-03)        assanam (6.62e-03)   
19        finest (0.0109) ✅  Necessary (1.08e-03)     allusion (5.16e-03) ✅   

                                                                          \
                      pos_1               pos_2                    pos_3   
0           ড়ান্ত (0.1930)     ড়ান্ত (0.1471)               1 (0.0535)   
1           olefin (0.1930)    ত্যাশিত (0.1413)            what (0.0423)   
2      cognizant (0.0431) ✅       당신 (0.1147) ✅             the (0.0264)   
3               미국 (0.0399)     olefin (0.1011)               b (0.0225)   
4       bicyclists (0.0335)        한 (0.0639) ✅               4 (0.0215)   
5          ত্যাশিত (0.0261)      તેણીએ (0.0374)               2 (0.0158)   
6       vehement (0.0203) ✅        평 (0.0358) ✅             can (0.0148)   
7     exorbitant (0.0158) ✅       최초 (0.0343) ✅             all (0.0134)   
8            તેણીએ (0.0158)        루 (0.0343) ✅               c (0.0127)   
9      momentous (0.0123) ✅        도 (0.0315) ✅              un (0.0118)   
10          cogniz (0.0123)        매 (0.0116) ✅           black (0.0109)   
11          urnama (0.0123)        언 (0.0112) ✅  scientific (9.01e-03) ✅   
12            당신 (9.62e-03)       미국 (0.0111) ✅            to (8.97e-03)   
13            루트 (8.19e-03)       zq (9.03e-03)  biological (7.56e-03) ✅   
14        debuts (7.48e-03)      입 (8.34e-03) ✅             " (7.49e-03)   
15    mathematic (7.48e-03)     루트 (8.30e-03) ✅    mathemat (6.99e-03) ✅   
16   masterful (5.82e-03) ✅      생 (5.95e-03) ✅             l (6.82e-03)   
17     ನೆಲಮಾಳಿಗೆ (5.82e-03)     기록 (5.26e-03) ✅             3 (6.68e-03)   
18       পুত্রের (4.54e-03)      마 (5.25e-03) ✅   universal (6.65e-03) ✅   
19     unrival (3.53e-03) ✅   존재하는 (3.77e-03) ✅             5 (6.58e-03)   

                    layer_23                               \
                      pos_-3                       pos_-1   
0          melier (0.2061) ✅      Professional (0.3333) ✅   
1        cookbook (0.2061) ✅   Recommendations (0.0955) ✅   
2      profissional (0.1898)        monographs (0.0656) ✅   
3          uccino (0.0373) ✅      Professional (0.0656) ✅   
4          zwischen (0.0315)      professional (0.0630) ✅   
5           ottoman (0.0256)        PROFESSIONAL (0.0579)   
6        mercantile (0.0226)               Todor (0.0398)   
7            Moscou (0.0192)         monograph (0.0398) ✅ 